In [ ]:
# duckdb allows us to run SQL queries directly on the parquet files
# without loading the entire dataset into pandas
# which would be slow and memory intensive

import duckdb
import pandas as pd
from pathlib import Path

In [ ]:
project_root = Path("/Users/saraneff/Desktop/taxi-project")

cleaned_dir = project_root / "data" / "cleaned"
raw_dir = project_root / "data" / "raw"

# this creates a duckdb connection object
# all SQL queries will be run through this connection
con = duckdb.connect()

In [ ]:
# duckdb has a built-in function read_parquet that can 
# read parquet files directly into a SQL query
# and treats parquet files like a table in a database

# * means all months, so I don't need to merge them into one file first
# con.execute sends the SQL query to duckdb 
# and fetchdf() returns the results as a pandas DataFrame
# .as_posix() converts the Path object to a string path 
# that duckdb can understand
# f means format string, so we can insert the 
# cleaned_dir path into the query
trips = con.execute(f"""
SELECT *
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
LIMIT 10
""").fetchdf()

# this gives us a quick peek at the data to make sure 
# everything is working
trips

In [ ]:
# taxi_zone_lookup.csv is a small file, so we can read it directly into pandas
# but we can also read it with duckdb to keep everything consistent
# it has the actual zone names and locations, which we will need later for analysis and visualization

zones = con.execute(f"""
SELECT *
FROM read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv')
""").fetchdf()

zones.head()

In [ ]:
# first query:
# how many trips started in each zone?
# as a proxy for how busy each zone is as a pickup location
# grouped by borough and zone because SQL requires all non-aggregated columns to be in the GROUP BY clause
trip_counts = con.execute(f"""
SELECT
    t.PULocationID,
    z.Borough,
    z.Zone,
    COUNT(*) AS trips
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet') t
LEFT JOIN read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID
GROUP BY
    t.PULocationID,
    z.Borough,
    z.Zone
ORDER BY trips DESC
""").fetchdf()

trip_counts.head(20)

In [ ]:
# total number of trips in the cleaned data
# .fetchall() returns the results as a list of tuples, 
# since we only have one row and one column, 
# we can just take the first element of the first tuple to get the count as an integer
# returns outer structure of list of tuples, where each tuple is a row of the result
con.execute(f"""
SELECT COUNT(*)
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
""").fetchall()

In [ ]:
# plot the busiest zones
# 1e6 = 1,000,000 to make the y-axis more readable
import matplotlib.pyplot as plt

trip_counts.head(15).plot(
    x="Zone",
    y="trips",
    kind="bar",
    figsize=(10,5),
    legend=False
)

plt.title("Top 15 Pickup Zones by Trip Count (2024)")
plt.ylabel("Trips")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# check the minimum fare amount in the cleaned data
con.execute(f"""
SELECT MIN(fare_amount) AS min_fare
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
""").fetchdf()

In [ ]:
# check the minimum trip distance in the cleaned data
con.execute(f"""
SELECT MIN(trip_distance) AS min_distance
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
""").fetchdf()

In [ ]:
# check the max trip duration in the cleaned data
con.execute(f"""
SELECT
MAX(tpep_dropoff_datetime - tpep_pickup_datetime) AS max_duration
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
""").fetchdf()

In [ ]:
# check how many trips have positive revenue

con.execute(f"""
SELECT COUNT(*) AS positive_revenue_trips
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
WHERE total_amount > 0
""").fetchdf()

In [ ]:
# distribution of trip durations
con.execute(f"""
SELECT
AVG(tpep_dropoff_datetime - tpep_pickup_datetime) AS avg_trip_time,
MAX(tpep_dropoff_datetime - tpep_pickup_datetime) AS max_trip_time
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
""").fetchdf()

In [ ]:
# average revenue per trip
con.execute(f"""
SELECT
AVG(total_amount) AS avg_revenue_per_trip
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
""").fetchdf()

In [ ]:
# number of trips originating in each zone during morning hours
# filters morning pickups between 6am and 9am, groups by pickup location, 
# joins the zone lookup, and counts the number of trips for each zone

morning_trip_counts = con.execute(f"""
SELECT
    t.PULocationID,
    z.Borough,
    z.Zone,
    COUNT(*) AS morning_trips
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet') t
LEFT JOIN read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID
WHERE
    EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 6 AND 9
GROUP BY
    t.PULocationID,
    z.Borough,
    z.Zone
ORDER BY morning_trips DESC
""").fetchdf()

morning_trip_counts.head(20)

In [ ]:
# plot the top 15 zones for morning pickups

import matplotlib.pyplot as plt

morning_trip_counts.head(15).plot(
    x="Zone",
    y="morning_trips",
    kind="bar",
    figsize=(10,5),
    legend=False
)

plt.title("Top 15 Pickup Zones (Morning Trips)")
plt.ylabel("Trips")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# calculate total revenue and revenue per hour for each zone during morning hours
# restricts to morning trips
# computes trip duration in hours
# computes revenue per hour by dividing total revenue by total hours
# this gives passenger revenue per occupied driving hour
# removes small zones with fewer than 2000 morning trips to focus on more significant areas

zone_earnings_morning = con.execute(f"""
SELECT
    t.PULocationID,
    z.Borough,
    z.Zone,
    COUNT(*) AS trips,

    SUM(total_amount) AS total_revenue,

    -- compute revenue per occupied hour
    -- extract epoch gives us the duration in seconds, so we divide by 3600 to get hours
    SUM(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600
    ) AS total_hours,

    SUM(total_amount) /
    SUM(EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600)
    AS revenue_per_hour

FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet') t

LEFT JOIN read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID

WHERE
    -- restrict to morning pickups
    EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 6 AND 9
    AND total_amount > 0
    AND PULocationID BETWEEN 1 AND 263
    AND DOLocationID BETWEEN 1 AND 263

GROUP BY
    t.PULocationID,
    z.Borough,
    z.Zone

HAVING COUNT(*) >= 2000
-- can adjust this threshold to focus on more or fewer zones

ORDER BY revenue_per_hour DESC
""").fetchdf()

zone_earnings_morning.head(20)

In [ ]:
# highest trip volume zones during morning hours
zone_earnings_morning.sort_values("trips", ascending=False).head(20)

In [ ]:
# plot the top zones by revenue/hour

import matplotlib.pyplot as plt

# df.plot() is a convenient way to create plots directly from a pandas DataFrame
# it automatically creates the figure, creates the axes, and plots the specified columns
# so plt.figure() is not needed here since df.plot() will create its own figure
zone_earnings_morning.head(15).plot(
    x="Zone",
    y="revenue_per_hour",
    kind="bar",
    figsize=(10,5),
    legend=False
)

plt.ylabel("Revenue per hour ($)")
plt.title("Top Pickup Zones by Morning Revenue Rate")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# scatter plot of revenue per hour vs trip volume for each zone
# since both revenue per hour and trip volume are important for understanding which zones are most lucrative,
# this plot can help us see if there is a relationship between the two and identify any outliers 
# that have high revenue per hour but low trip volume, or vice versa
# alpha controls the transparency of the points, so we can see overlapping points more clearly

import matplotlib.pyplot as plt

# plt.figure() creates a new figure for the plot, and figsize sets the size of the figure in inches
# it automatically creates an axes object for us to plot on, so we can just call plt.scatter() directly
plt.figure(figsize=(10,6))

plt.scatter(
    zone_earnings_morning["trips"],
    zone_earnings_morning["revenue_per_hour"],
    alpha=0.6
)

plt.xlabel("Number of Morning Trips")
plt.ylabel("Revenue per Hour ($)")
plt.title("Revenue per Hour vs Trip Volume by Pickup Zone (6–10 AM)")

plt.show()

In [ ]:
# define the top zones based on revenue per hour to label them on the scatter plot
top_zones = zone_earnings_morning.nlargest(10, "revenue_per_hour")

plt.figure(figsize=(10,6))

plt.scatter(
    zone_earnings_morning["trips"],
    zone_earnings_morning["revenue_per_hour"],
    alpha=0.4
)

# label the top zones on the scatter plot with their names
# iterrows() returns two things (index, row) for each row in the DataFrame, 
# we only care about the row here so we use _ for the index
# plt.text(x, y, text) adds text at the specified x and y coordinates on the plot
for _, row in top_zones.iterrows():
    plt.text(
        row["trips"],
        row["revenue_per_hour"],
        row["Zone"],
        fontsize=8
    )

plt.xlabel("Number of Morning Trips")
plt.ylabel("Revenue per Hour ($)")
plt.title("Revenue vs Demand by Pickup Zone (6–10 AM)")

plt.show()

In [ ]:
# now we want to highlight the zone with the highest trip volume
# since it also has decently high revenue per hour
# idxmax() gives us the index of the row with the maximum value in the "trips" column
max_zone = zone_earnings_morning.loc[
    zone_earnings_morning["trips"].idxmax()
]

plt.figure(figsize=(10,6))

plt.scatter(
    zone_earnings_morning["trips"],
    zone_earnings_morning["revenue_per_hour"],
    alpha=0.4
)

# s controls the size of the marker
plt.scatter(
    max_zone["trips"],
    max_zone["revenue_per_hour"],
    color="red",
    s=100
)

plt.text(
    max_zone["trips"],
    max_zone["revenue_per_hour"],
    max_zone["Zone"],
)

plt.xlabel("Number of Morning Trips")
plt.ylabel("Revenue per Hour ($)")
plt.title("Revenue vs Demand by Pickup Zone (6–10 AM)")

plt.show()

In [ ]:
# now we are using a combined earnings index
# that accounts for both revenue per hour and trip volume
# to identify zones that are strong performers on both dimensions, 
# rather than just one or the other. This can help highlight zones that
# are not only high-revenue but also have enough demand to be attractive for drivers.

# how the earnings index is calculated:
# earnings_index = revenue_per_hour * (trips / 4)
# assumptions used
# observed trips represent arrival rate of passengers
# average trip duration is consistent across zones
# idle time is inversely related to trip frequency
# drivers pick up passengers where they drop off
# occupied time approximates working time
# morning pickups approximate shift starts
# historical patterns represent future conditions

zone_earnings_morning = con.execute(f"""
SELECT
    t.PULocationID,
    z.Borough,
    z.Zone,

    COUNT(*) AS trips,

    -- revenue per occupied hour
    SUM(total_amount) /
    SUM(EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600)
    AS revenue_per_hour,

    -- demand intensity
    -- divide by 4 because we are looking at a 4-hour morning window (6am-10am)
    COUNT(*) / 4.0 AS trips_per_hour,

    -- combined earnings index
    (
        SUM(total_amount) /
        SUM(EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600)
    ) * (COUNT(*) / 4.0)
    AS earnings_index

FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet') t

LEFT JOIN read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID

-- remove EWR since it's an outlier with very high revenue per hour but low trip volume,
-- and we want to focus on zones with more typical demand patterns
-- it also has a different demand pattern since it's an airport zone, 
-- so it would skew the analysis of typical zones
WHERE
    EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 6 AND 9
    AND total_amount > 0
    AND z.Borough != 'EWR'
    AND t.PULocationID BETWEEN 1 AND 263
    AND t.DOLocationID BETWEEN 1 AND 263

GROUP BY
    t.PULocationID,
    z.Borough,
    z.Zone

HAVING COUNT(*) >= 2000

ORDER BY earnings_index DESC
""").fetchdf()

In [ ]:
# load taxi zone shapefile
# geopandas is a library that extends pandas to work with geospatial data
import geopandas as gpd

# .shp file contains the actual geometries of the taxi zones, which we can use for mapping and spatial analysis
# .shp file is just the geometry, we also need the .dbf file for the attributes like zone name and ID, 
# but geopandas can read them all together
zones_map = gpd.read_file("../data/raw/taxi_zones/taxi_zones.shp")

zones_map.head()

In [ ]:
# merge earnings data with zone shapefile for mapping
map_data = zones_map.merge(
    zone_earnings_morning,
    left_on="LocationID",
    right_on="PULocationID",
    how="left"
)

map_data.head()

In [ ]:
# plot revenue per hour on a map
import matplotlib.pyplot as plt

# plt.subplots() returns two things: the figure object and the axes object.
# the figure is the overall container for the plot, and the axes is where the actual plotting happens. 
# Since we only have one plot, we can just use ax to refer to the axes.
# 1,1 means 1 row and 1 column of plots, figsize sets the size of the overall figure
# earlier we relied on matplotlib's default figure size, but since we are plotting a map with many zones,
# we can increase the figure size to make it more readable and visually appealing
fig, ax = plt.subplots(1, 1, figsize=(12,10))

# map_data is a geodataframe that has a geometry column with the shapes of the zones, 
# which geopandas requires for plotting the zones on a map,
# and a revenue_per_hour column with the values we want to plot
# column specifies which column to use for coloring the zones, cmap specifies the color map,
# viridis is a graient that is commonly used for continuous data and is colorblind-friendly
# linewidth and edgecolor add borders to the zones, legend=True adds a colorbar legend, 
# and ax=ax specifies which axes to plot on
# doesn't create a new figure because we explcitly specify the axes to plot on with ax=ax, 
# so it will use the figure we created with plt.subplots() and plot on that
map_data.plot(
    column="revenue_per_hour",
    cmap="viridis",
    linewidth=0.5,
    edgecolor="black",
    legend=True,
    ax=ax
)

ax.set_title("Taxi Revenue per Hour by Pickup Zone (6–10 AM)", fontsize=14)
ax.axis("off")

plt.show()

In [ ]:
# now plot the earnings index instead of revenue per hour
fig, ax = plt.subplots(1, 1, figsize=(12,10))

map_data.plot(
    column="earnings_index",
    cmap="plasma",
    linewidth=0.5,
    edgecolor="black",
    legend=True,
    ax=ax
)

ax.set_title("Taxi Earnings Opportunity by Zone (6–10 AM)", fontsize=14)
ax.axis("off")

plt.show()


In [ ]:
# add label for the top zone by earnings index

fig, ax = plt.subplots(1, 1, figsize=(12,10))

map_data.plot(
    column="earnings_index",
    cmap="plasma",
    linewidth=0.5,
    edgecolor="black",
    legend=True,
    ax=ax
)

# Select top zones by earnings opportunity
top_zones = map_data.nlargest(1, "earnings_index")

# Add labels at zone centroids
for idx, row in top_zones.iterrows():
    x = row.geometry.centroid.x
    y = row.geometry.centroid.y
    
    ax.text(
        x,
        y,
        row["Zone"],   # zone name
        fontsize=8,
        ha="center"
    )

ax.set_title("Taxi Earnings Opportunity by Zone (6–10 AM)", fontsize=14)
ax.axis("off")

plt.show()

In [ ]:
# measure how often drivers remain in the same zone for their next pickup 
# after dropping off a passenger
zone_transitions = con.execute(f"""
SELECT
    PULocationID,
    COUNT(*) AS trips,
    SUM(CASE WHEN PULocationID = DOLocationID THEN 1 ELSE 0 END) AS same_zone_trips,
    SUM(CASE WHEN PULocationID = DOLocationID THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prob_same_zone
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
WHERE EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 6 AND 9
AND PULocationID BETWEEN 1 AND 263
AND DOLocationID BETWEEN 1 AND 263
GROUP BY PULocationID
HAVING COUNT(*) >= 2000
ORDER BY prob_same_zone DESC
""").fetchdf()

In [ ]:
zone_transitions.head(10)

In [ ]:
# plot the probability of remaining in the same zone for the next trip by pickup zone

plt.figure(figsize=(12,6))

plt.bar(
    zone_transitions["PULocationID"],
    zone_transitions["prob_same_zone"]
)

plt.xlabel("Pickup Zone (LocationID)")
plt.ylabel("Probability of Same-Zone Next Trip")
plt.title("Probability of Remaining in Same Zone After Trip")

plt.show()

In [ ]:
# add zone names to the transitions data
import pandas as pd

zone_lookup = pd.read_csv("../data/raw/taxi_zone_lookup.csv")

zone_transitions = zone_transitions.merge(
    zone_lookup,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

zone_transitions.head(10)